# Prompt Testing Workbench

Use this notebook to:
1. Load a prompt from YAML
2. Inspect its variables
3. Run it with test inputs
4. Modify the prompt (create v2)
5. Run again and compare outputs

In [1]:
import sys
sys.path.insert(0, "..")

from utils.prompt_loader import load_prompt, fill_prompt, get_prompt_variables, list_prompt_keys, list_all_prompts
from utils.api_client import call_model
from utils.prompt_runner import run_prompt, run_test_cases, compare_versions, save_results

## 0. Fetch Real Inputs from Mangowater

Run a query on the live deployment first, then fetch the exact inputs each prompt received.
These become your test fixtures — real data, no guessing.

In [4]:
import requests
import json

# Step 1: Set the mangowater DeepSearch URL
NORA = "https://deepsearch.mangowater-31feba87.swedencentral.azurecontainerapps.io"  # <-- adjust if different

# Step 2: Fetch recent prompt logs (after you've run a query on the platform)
response = requests.get(f"{NORA}/api/v1/debug/prompt-logs", params={"limit": 50})
logs = response.json()

print(f"Found {logs['count']} prompt log entries\n")

# Step 3: Browse what's available
for i, entry in enumerate(logs["logs"]):
    print(f"[{i}] {entry['category']}/{entry['prompt_key']}  ({entry['rendered_length']} chars)  {entry['timestamp_iso']}")
    print(f"     Variables: {list(entry['variables'].keys())}")
    print()

Found 7 prompt log entries

[0] supervisor/generate_concise_answer  (2063 chars)  2026-03-23T11:53:04.310360+00:00
     Variables: ['query', 'search_summary', 'max_sentences']

[1] supervisor/generate_report_factual  (5916 chars)  2026-03-23T11:52:58.848887+00:00
     Variables: ['query', 'search_findings', 'sources_count']

[2] supervisor/assess_quality  (2731 chars)  2026-03-23T11:52:53.491700+00:00
     Variables: ['query', 'answer', 'sources_count', 'confidence']

[3] confluence_agent/system  (9364 chars)  2026-03-23T11:52:42.268280+00:00
     Variables: ['agent_name', 'tool_count', 'tool_names']

[4] supervisor/search_query  (807 chars)  2026-03-23T11:52:40.551152+00:00
     Variables: ['query', 'query_type', 'complexity', 'format_hint']

[5] supervisor/system  (3223 chars)  2026-03-23T11:52:40.551124+00:00
     Variables: ['agent_count', 'agent_descriptions', 'tool_count']

[6] supervisor/classify_query  (981 chars)  2026-03-23T11:52:38.811043+00:00
     Variables: ['query']



In [5]:
# Step 4: Pick a log entry and use its real variables
LOG_INDEX = 0  # <-- change this to pick a different entry

entry = logs["logs"][LOG_INDEX]
print(f"Using: {entry['category']}/{entry['prompt_key']}")
print(f"Timestamp: {entry['timestamp_iso']}")
print(f"\nVariables captured:")
for k, v in entry["variables"].items():
    preview = v[:200] + "..." if len(v) > 200 else v
    print(f"  {k}: {preview}")

# These are now ready to use in sections below
real_variables = entry["variables"]
real_category = entry["category"]
real_prompt_key = entry["prompt_key"]

Using: supervisor/generate_concise_answer
Timestamp: 2026-03-23T11:53:04.310360+00:00

Variables captured:
  query: Show me the Horsch Hour logs for both Omar and Lisa In a table format
  search_summary: Here is the Horsch Hour log for Omar and Lisa, detailing their activities, dates, and hours:

| **Day**     | **Person** | **Task**                                                                     ...
  max_sentences: 4


## 1. Explore Available Prompts

In [6]:
# List all prompt files and their keys
all_prompts = list_all_prompts("../prompts")
for file, keys in all_prompts.items():
    print(f"\n📄 {file}")
    for k in keys:
        print(f"   - {k}")


📄 deepgram/evidence_evaluation.yaml
   - system
   - user

📄 deepgram/patterns.yaml
   - greeting
   - self_intro
   - self_intro_first_name
   - quotation

📄 deepsearch/confluence_agent.yaml
   - system
   - synthesize_findings

📄 deepsearch/elasticsearch_agent.yaml
   - system
   - search_strategy
   - synthesize_findings
   - evaluate_results

📄 deepsearch/jira_agent.yaml
   - system
   - synthesize_findings

📄 deepsearch/query_reformulator.yaml
   - reformulate_query
   - reformulate_for_tickets
   - reformulate_for_docs
   - reformulate_for_german_technical
   - reformulate_multi_query

📄 deepsearch/retriever.yaml
   - generate_answer
   - estimate_confidence

📄 deepsearch/supervisor.yaml
   - system
   - classify_query
   - search_query
   - search_query_with_history
   - assess_quality
   - assess_agent_result
   - generate_report
   - generate_report_factual
   - generate_report_troubleshooting
   - generate_report_procedural
   - generate_report_comparison
   - generate_conci

## 2. Load a Specific Prompt

In [ ]:
# Option A: Auto-load from Section 0 (uses the prompt you picked from mangowater logs)
# Maps category names to local YAML files
CATEGORY_TO_FILE = {
    "supervisor": "../prompts/deepsearch/supervisor.yaml",
    "confluence_agent": "../prompts/deepsearch/confluence_agent.yaml",
    "jira_agent": "../prompts/deepsearch/jira_agent.yaml",
    "elasticsearch_agent": "../prompts/deepsearch/elasticsearch_agent.yaml",
    "websearch_agent": "../prompts/deepsearch/websearch_agent.yaml",
    "retriever": "../prompts/deepsearch/retriever.yaml",
    "query_reformulator": "../prompts/deepsearch/query_reformulator.yaml",
}

# Auto-set from Section 0 if available
try:
    PROMPT_FILE = CATEGORY_TO_FILE.get(real_category, f"../prompts/deepsearch/{real_category}.yaml")
    PROMPT_KEY = real_prompt_key
    variables = real_variables
    print(f"Auto-loaded from mangowater: {real_category}/{real_prompt_key}")
except NameError:
    # Option B: Manual override (if you skipped Section 0)
    PROMPT_FILE = "../prompts/deepsearch/supervisor.yaml"
    PROMPT_KEY = "generate_report_factual"
    variables = {
        "query": "Welche Richtlinien gelten für Biegeteile?",
        "search_findings": "Source 1: Mindestbiegeradius 1x bei S235.",
        "sources_count": "1",
    }
    print(f"Manual mode: {PROMPT_KEY}")

from utils.prompt_loader import get_template
prompt = load_prompt(PROMPT_FILE)
template = get_template(prompt, PROMPT_KEY)

print(f"\nPrompt file: {PROMPT_FILE}")
print(f"Prompt key: {PROMPT_KEY}")
print(f"Variables: {list(variables.keys())}")
print(f"Template length: {len(template)} chars")
print("---")
print(template[:500] + "..." if len(template) > 500 else template)

## 3. Run Baseline (v1)

In [ ]:
# Variables are already set from Section 2 (auto from mangowater or manual)
# Preview what will be sent to the LLM
filled = fill_prompt(template, variables)
print(f"=== FILLED PROMPT ({len(filled)} chars) ===")
print(filled[:1000] + "..." if len(filled) > 1000 else filled)

In [ ]:
# Run v1 baseline — choose your model
TARGET_MODEL = "gpt-4o"  # or "mistral-25b"

result_v1 = run_prompt(prompt, variables, prompt_key=PROMPT_KEY, target_model=TARGET_MODEL)

print(f"Model: {result_v1['target_model']} | Duration: {result_v1['duration_seconds']}s")
print("=== OUTPUT v1 ===")
print(result_v1["output"])

## 4. Create v2 (Modified Prompt)

Copy the YAML file, make your changes, and save as a new version.
For example: `prompts/deepsearch/supervisor_v2.yaml`

In [ ]:
# Load the modified prompt
PROMPT_FILE_V2 = "../prompts/deepsearch/supervisor_v2.yaml"  # <-- create this file with your changes

prompt_v2 = load_prompt(PROMPT_FILE_V2)
template_v2 = prompt_v2[PROMPT_KEY]

print(f"v2 length: {len(template_v2)} chars")
print(f"v2 variables: {get_prompt_variables(template_v2)}")

In [ ]:
# Run v2 with the SAME variables and model
result_v2 = run_prompt(prompt_v2, variables, prompt_key=PROMPT_KEY, target_model=TARGET_MODEL)

print(f"Model: {result_v2['target_model']} | Duration: {result_v2['duration_seconds']}s")
print("=== OUTPUT v2 ===")
print(result_v2["output"])

## 5. Compare Outputs

In [ ]:
print("=" * 80)
print("INPUT")
print("=" * 80)
for k, v in variables.items():
    preview = str(v)[:150] + "..." if len(str(v)) > 150 else str(v)
    print(f"  {k}: {preview}")

print("\n" + "=" * 80)
print(f"OUTPUT v1 (baseline) — {result_v1['duration_seconds']}s")
print("=" * 80)
print(result_v1["output"])

print("\n" + "=" * 80)
print(f"OUTPUT v2 (improved) — {result_v2['duration_seconds']}s")
print("=" * 80)
print(result_v2["output"])

## 6. Batch Run (once you have test cases)

After documenting test cases in Excel and exporting to JSON, use this section.

In [ ]:
import json

# Load test cases from JSON (export from your Excel file)
# with open("../test_cases/your_test_cases.json") as f:
#     test_cases = json.load(f)

# results_v1 = run_test_cases(prompt, test_cases, prompt_key=PROMPT_KEY)
# results_v2 = run_test_cases(prompt_v2, test_cases, prompt_key=PROMPT_KEY)

# comparison = compare_versions(results_v1, results_v2)
# print(comparison)

# Save results
# save_results(results_v1, "../results/v1_results.json")
# save_results(results_v2, "../results/v2_results.json")

## 7. Compare Models (same prompt, different LLMs)

Run the same prompt on GPT-4o and Mistral 25B side by side.

In [ ]:
from utils.prompt_runner import compare_models

# Uses the same prompt + variables from Section 2, runs on both models
result = compare_models(
    prompt, variables,
    prompt_key=PROMPT_KEY,
    models=["gpt-4o", "mistral-25b"]
)

print(result["comparison"])